In [97]:
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
import os

import numpy as np
import pandas as pd
import rasterio
from skimage.metrics import (
    structural_similarity,
    peak_signal_noise_ratio,
)


def compute_metrics(original_path, cleaned_path):
    """
    Calcula SSIM e PSNR entre duas imagens Sentinel-2.

    Returns
    -------
    dict
    """

    with rasterio.open(original_path) as src1:
        img1 = src1.read(masked=True).astype(np.float32)

    with rasterio.open(cleaned_path) as src2:
        img2 = src2.read(masked=True).astype(np.float32)

    if img1.shape != img2.shape:
        raise ValueError(
            f"Shape diferente:\n"
            f"{original_path}\n"
            f"{cleaned_path}"
        )

    n_bands = img1.shape[0]

    ssim_values = []
    psnr_values = []

    for band in range(n_bands):

        original = img1[band]
        cleaned = img2[band]

        mask = ~(original.mask | cleaned.mask)

        if mask.sum() == 0:
            continue

        original = original.filled(np.nan)
        cleaned = cleaned.filled(np.nan)

        valid_original = original[mask]
        valid_cleaned = cleaned[mask]

        data_range = (
            max(valid_original.max(), valid_cleaned.max())
            - min(valid_original.min(), valid_cleaned.min())
        )

        if data_range == 0:
            data_range = 1.0

        img_original = np.zeros_like(original, dtype=np.float32)
        img_cleaned = np.zeros_like(cleaned, dtype=np.float32)

        img_original[mask] = valid_original
        img_cleaned[mask] = valid_cleaned

        ssim_band = structural_similarity(
            img_original,
            img_cleaned,
            data_range=data_range,
        )

        psnr_band = peak_signal_noise_ratio(
            valid_original,
            valid_cleaned,
            data_range=data_range,
        )

        ssim_values.append(ssim_band)
        psnr_values.append(psnr_band)

    return {
        "original": original_path,
        "cleaned": cleaned_path,
        "ssim": float(np.mean(ssim_values)),
        "psnr": float(np.mean(psnr_values)),
    }


def evaluate_images(
    original_paths,
    cleaned_paths,
    max_workers=os.cpu_count(),
):
    """
    Avalia vários pares de imagens em paralelo.

    Parameters
    ----------
    original_paths : list[str]
    cleaned_paths : list[str]

    Returns
    -------
    pandas.DataFrame
    """

    if len(original_paths) != len(cleaned_paths):
        raise ValueError("As listas devem possuir o mesmo tamanho.")

    pairs = list(zip(original_paths, cleaned_paths))

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        futures = [
            executor.submit(compute_metrics, original, cleaned)
            for original, cleaned in zip(original_paths, cleaned_paths)
        ]

        results = [f.result() for f in futures]

    return pd.DataFrame(results)

In [98]:
images = pd.read_csv("../data/temporal_interpolation_images.csv")

In [99]:
images.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806 entries, 0 to 805
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   mask                            806 non-null    object 
 1   cloud_percentage                806 non-null    float64
 2   cloud_shadow_percentage         806 non-null    float64
 3   reservoir                       806 non-null    object 
 4   image_path                      806 non-null    object 
 5   random_mask                     806 non-null    object 
 6   random_cloud_percentage         806 non-null    float64
 7   random_cloud_shadow_percentage  806 non-null    float64
dtypes: float64(4), object(4)
memory usage: 50.5+ KB


In [100]:
original_paths = images["image_path"].tolist()

In [101]:
original_paths[:5]

['D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190717.tif',
 'D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190910.tif',
 'D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20191124.tif',
 'D:/GeoPipe/data/02_boa_images/argemiro\\2020\\sentinel_BOA_S2_SR_argemiro_20200323.tif']

In [102]:
import glob 
cleaned_images = glob.glob("../data/temp/interpolation/clean_images/**/**/**/**/*.tif")

In [103]:
cleaned_images[:5]

['../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20190607_clean.tif',
 '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20190717_clean.tif',
 '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20190910_clean.tif',
 '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20191124_clean.tif',
 '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2020\\sentinel_BOA_S2_SR_argemiro_20200323_clean.tif']

In [104]:
import re


def extrair_localidade_e_data(caminho):
    # Procura por qualquer texto entre o último caractere '_' e a sequência de 8 dígitos da data
    # Exemplo: ..._argemiro_20191124.tif -> Grupo 1: argemiro, Grupo 2: 20191124
    match = re.search(r'([a-zA-Z0-9\-]+)_(\d{8})', caminho)
    if match:
        localidade = match.group(1)
        data = match.group(2)
        return (localidade, data)  # Retorna uma tupla como chave identificadora única
    return None

# 1. Cria o dicionário usando a combinação (localidade, data) como chave única
mapa_clean = {}
for img in cleaned_images:
    chave = extrair_localidade_e_data(img)
    if chave:
        mapa_clean[chave] = img

# 2. Faz o cruzamento preciso comparando localidade + data
pares_confirmados = []
for img_orig in original_paths:
    chave_orig = extrair_localidade_e_data(img_orig)
    if chave_orig in mapa_clean:
        pares_confirmados.append((img_orig, mapa_clean[chave_orig]))

# Exibindo os resultados pareados corretamente
for original, clean in pares_confirmados:
    print(f"Original: {original}")
    print(f"Clean   : {clean}\n" + "-"*50)

Original: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190607.tif
Clean   : ../data/temp/interpolation/clean_images\argemiro\fmask\temporal_interpolation\2019\sentinel_BOA_S2_SR_argemiro_20190607_clean.tif
--------------------------------------------------
Original: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190717.tif
Clean   : ../data/temp/interpolation/clean_images\argemiro\fmask\temporal_interpolation\2019\sentinel_BOA_S2_SR_argemiro_20190717_clean.tif
--------------------------------------------------
Original: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20190910.tif
Clean   : ../data/temp/interpolation/clean_images\argemiro\fmask\temporal_interpolation\2019\sentinel_BOA_S2_SR_argemiro_20190910_clean.tif
--------------------------------------------------
Original: D:/GeoPipe/data/02_boa_images/argemiro\2019\sentinel_BOA_S2_SR_argemiro_20191124.tif
Clean   : ../data/temp/interpolation/clean_imag

In [105]:
len(pares_confirmados)

806

In [106]:
pares_confirmados[0]

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20190607_clean.tif')

In [107]:
# spliting the confirmed pairs into two separate lists for evaluation
original_confirmed = [pair[0] for pair in pares_confirmados]
cleaned_confirmed = [pair[1] for pair in pares_confirmados]

In [108]:
df = evaluate_images(
    original_confirmed,
    cleaned_confirmed,
)

d:\GeoPipe\.venv_win\Lib\site-packages\skimage\metrics\simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


In [109]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806 entries, 0 to 805
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   original  806 non-null    object 
 1   cleaned   806 non-null    object 
 2   ssim      806 non-null    float64
 3   psnr      806 non-null    float64
dtypes: float64(2), object(2)
memory usage: 25.3+ KB


In [110]:
df.describe()

d:\GeoPipe\.venv_win\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,ssim,psnr
count,806.000000,806.000000
mean,0.917526,inf
std,0.084150,NaN
min,0.466927,17.899336
25%,0.880634,27.490287
50%,0.947634,32.438441
75%,0.978475,37.363952
max,1.000000,inf


In [111]:
# remove 2026xxyy date images by name
df = df[~df['original'].str.contains(r'2026\d{4}')]

In [112]:
final_df = df.merge(images, left_on='original', right_on='image_path')

In [113]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 757 entries, 0 to 756
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   original                        757 non-null    object 
 1   cleaned                         757 non-null    object 
 2   ssim                            757 non-null    float64
 3   psnr                            757 non-null    float64
 4   mask                            757 non-null    object 
 5   cloud_percentage                757 non-null    float64
 6   cloud_shadow_percentage         757 non-null    float64
 7   reservoir                       757 non-null    object 
 8   image_path                      757 non-null    object 
 9   random_mask                     757 non-null    object 
 10  random_cloud_percentage         757 non-null    float64
 11  random_cloud_shadow_percentage  757 non-null    float64
dtypes: float64(6), object(6)
memory usag

In [114]:
final_df.head()

,original,cleaned,ssim,psnr,mask,cloud_percentage,cloud_shadow_percentage,reservoir,image_path,random_mask,random_cloud_percentage,random_cloud_shadow_percentage
0,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,../data/temp/interpolation/clean_images\argemi...,0.852352,29.061439,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,0.998422,1.785773,argemiro,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,34.043427,12.213716
1,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,../data/temp/interpolation/clean_images\argemi...,0.832289,28.059797,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,0.406730,0.759103,argemiro,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,27.222952,6.650423
2,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,../data/temp/interpolation/clean_images\argemi...,0.922486,29.810493,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,5.637656,3.893343,argemiro,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,9.884191,4.242571
3,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,../data/temp/interpolation/clean_images\argemi...,0.719432,24.468731,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,0.005788,0.463728,argemiro,D:/GeoPipe/data/02_boa_images/argemiro\2019\se...,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,47.161575,13.531294
4,D:/GeoPipe/data/02_boa_images/argemiro\2020\se...,../data/temp/interpolation/clean_images\argemi...,0.864947,26.918130,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,1.403831,0.879455,argemiro,D:/GeoPipe/data/02_boa_images/argemiro\2020\se...,D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\...,42.058675,4.901957


In [115]:
final_df.iloc[:1, :].to_dict()

{'original': {0: 'D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif'},
 'cleaned': {0: '../data/temp/interpolation/clean_images\\argemiro\\fmask\\temporal_interpolation\\2019\\sentinel_BOA_S2_SR_argemiro_20190607_clean.tif'},
 'ssim': {0: 0.8523518241055227},
 'psnr': {0: 29.06143863823678},
 'mask': {0: 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2019\\mask_sentinel_TOA_S2_argemiro_20190607.tif'},
 'cloud_percentage': {0: 0.9984221532700686},
 'cloud_shadow_percentage': {0: 1.785772704509823},
 'reservoir': {0: 'argemiro'},
 'image_path': {0: 'D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif'},
 'random_mask': {0: 'D:/GeoPipe/data/03_cloud_masks/argemiro/fmask\\2025\\mask_sentinel_TOA_S2_argemiro_20251209.tif'},
 'random_cloud_percentage': {0: 34.043427274351565},
 'random_cloud_shadow_percentage': {0: 12.21371644559197}}

In [116]:
final_df.drop(columns=['image_path'], inplace=True)

In [117]:
final_df.to_csv("../data/temporal_interpolation_metrics.csv", index=False)

## Analysis

In [118]:
final_df["cloud_percentage"].corr(final_df["ssim"])

-0.21529164852396945

In [119]:
final_df["cloud_percentage"].corr(final_df["psnr"])


d:\GeoPipe\.venv_win\Lib\site-packages\numpy\lib\function_base.py:2742: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]


nan

In [120]:
mask = np.isfinite(final_df["psnr"])

r = final_df.loc[mask, "cloud_percentage"].corr(
        final_df.loc[mask, "psnr"]
)

print(r)

-0.34132316162871407


In [121]:
df.assign(
    psnr=final_df["psnr"].replace([np.inf, -np.inf], np.nan)
).describe()

,ssim,psnr
count,753.000000,725.000000
mean,0.915765,33.126052
std,0.085559,7.708382
min,0.466927,17.899336
25%,0.876709,27.524845
50%,0.946314,32.208829
75%,0.977007,37.165713
max,1.000000,78.256813


In [145]:
import glob

# Padrão para 2017-2019 e padrão para 2020-2025
padrao_10 = "D:/GeoPipe/data/04_clean_images/**/fmask/**/2019/*.tif"
padrao_20 = "D:/GeoPipe/data/04_clean_images/**/fmask/**/202[0-5]/*.tif"

# Junta os resultados das duas buscas
images_2017_2025 = glob.glob(padrao_10, recursive=True) + glob.glob(padrao_20, recursive=True)


In [146]:
len(images_2017_2025)

3170

In [144]:
images_2017_2025[-1]

'D:/GeoPipe/data/04_clean_images\\sume\\fmask\\temporal_interpolation\\2025\\sentinel_BOA_S2_SR_sume_20251230_clean.tif'

In [148]:
final_df['original'].to_list()[0], final_df['original'].to_list()[-1]

('D:/GeoPipe/data/02_boa_images/argemiro\\2019\\sentinel_BOA_S2_SR_argemiro_20190607.tif',
 'D:/GeoPipe/data/02_boa_images/sume\\2025\\sentinel_BOA_S2_SR_sume_20251230.tif')